# Agentic Healthcare Assistant

A virtual medical assistant that reads a request in natural language and routes it to the right tool: patient records, history retrieval, AI summaries, appointment booking or disease information.

**To run:** set the `OPENAI_API_KEY` environment variable, or run the cell below and paste your key in the app's *API Key* tab. The key is never stored in this notebook.

> Educational prototype with in-memory data. Not a medical device and not medical advice.


In [ ]:
# ============================================================
# STEP 1: Install required libraries
# ============================================================

!pip install -q gradio openai


# ============================================================
# STEP 2: Import required libraries
# ============================================================

import os
import gradio as gr
from openai import OpenAI
import datetime


# ============================================================
# STEP 3: OpenAI API key (never hard-coded)
# ============================================================
# The key is read from the OPENAI_API_KEY environment variable if it is set.
# Otherwise, paste it into the "API Key" tab of the app at runtime.
# Never paste the key directly into this notebook.

client = None

if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()


def save_api_key(api_key):
    global client

    if api_key is None or api_key.strip() == "":
        return "Please enter your OpenAI API key."

    client = OpenAI(api_key=api_key.strip())

    return "API key saved for this session."


def key_missing_message():
    return "Please enter and save your OpenAI API key in the 'API Key' tab first."


# ============================================================
# STEP 4: Create simple memory/database
# ============================================================
# In real healthcare systems, this would be replaced by:
# - Hospital database
# - EHR database
# - Doctor calendar API
# - Patient management system

patients_db = {}
appointments_db = []
agent_logs = []


# ============================================================
# STEP 5: Agent log function
# ============================================================

def add_log(message):
    time_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    agent_logs.append(f"[{time_now}] {message}")


# ============================================================
# STEP 6: Add or update patient record
# ============================================================

def add_patient_record(patient_name, age, condition, history):
    if patient_name is None or patient_name.strip() == "":
        return "Please enter the patient name."

    if condition is None or condition.strip() == "":
        return "Please enter the medical condition."

    if history is None or history.strip() == "":
        return "Please enter the medical history."

    patients_db[patient_name.lower()] = {
        "name": patient_name,
        "age": age,
        "condition": condition,
        "history": history
    }

    add_log(f"Patient record added or updated for {patient_name}")

    return f"Patient record saved successfully for {patient_name}."


# ============================================================
# STEP 7: Retrieve patient history
# ============================================================

def get_patient_history(patient_name):
    if patient_name is None or patient_name.strip() == "":
        return "Please enter the patient name."

    patient = patients_db.get(patient_name.lower())

    if not patient:
        add_log(f"No patient record found for {patient_name}")
        return "No patient record found. Please add the patient record first."

    add_log(f"Retrieved medical history for {patient_name}")

    return f"""
Patient Name: {patient['name']}
Age: {patient['age']}
Condition: {patient['condition']}
Medical History: {patient['history']}
"""


# ============================================================
# STEP 8: Summarize patient history using AI
# ============================================================

def summarize_patient_history(patient_name):
    if patient_name is None or patient_name.strip() == "":
        return "Please enter the patient name."

    patient = patients_db.get(patient_name.lower())

    if not patient:
        add_log(f"No record found for summary request: {patient_name}")
        return "No patient record found. Please add the patient record first."

    if client is None:
        return key_missing_message()

    prompt = f"""
You are a safe healthcare assistant.

Summarize this patient's medical history in very simple language.

Patient Name: {patient['name']}
Age: {patient['age']}
Condition: {patient['condition']}
Medical History: {patient['history']}

Rules:
- Keep the summary simple.
- Do not give a final medical diagnosis.
- Do not prescribe medicine.
- Mention that the patient should consult a qualified doctor.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a safe healthcare assistant. Do not provide final medical diagnosis or prescriptions."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        add_log(f"Generated medical history summary for {patient_name}")

        return response.choices[0].message.content

    except Exception as e:
        add_log(f"Error while summarizing patient history: {str(e)}")
        return f"Error while generating summary: {str(e)}"


# ============================================================
# STEP 9: Book doctor appointment
# ============================================================

def book_appointment(patient_name, doctor_type, preferred_date):
    if patient_name is None or patient_name.strip() == "":
        return "Please enter the patient name."

    if doctor_type is None or doctor_type.strip() == "":
        return "Please select doctor type."

    if preferred_date is None or preferred_date.strip() == "":
        return "Please enter the preferred date."

    appointment = {
        "patient_name": patient_name,
        "doctor_type": doctor_type,
        "date": preferred_date,
        "status": "Booked"
    }

    appointments_db.append(appointment)

    add_log(f"Appointment booked for {patient_name} with {doctor_type} on {preferred_date}")

    return f"""
Appointment booked successfully.

Patient Name: {patient_name}
Doctor Type: {doctor_type}
Preferred Date: {preferred_date}
Status: Booked
"""


# ============================================================
# STEP 10: Show all booked appointments
# ============================================================

def show_appointments():
    if len(appointments_db) == 0:
        return "No appointments booked yet."

    result = ""

    for index, appointment in enumerate(appointments_db, start=1):
        result += f"""
Appointment {index}
Patient Name: {appointment['patient_name']}
Doctor Type: {appointment['doctor_type']}
Date: {appointment['date']}
Status: {appointment['status']}

"""

    add_log("Displayed appointment list")

    return result


# ============================================================
# STEP 11: Disease information using AI
# ============================================================

def disease_information(disease_name):
    if disease_name is None or disease_name.strip() == "":
        return "Please enter the disease name."

    if client is None:
        return key_missing_message()

    prompt = f"""
Explain this medical topic in very simple language:

Topic: {disease_name}

Include:
1. What it is
2. Common symptoms
3. General treatment options
4. When to consult a doctor

Important safety rules:
- Do not provide personal diagnosis.
- Do not prescribe medicine.
- Do not replace a doctor.
- Clearly mention that the user should consult a qualified doctor.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a safe healthcare information assistant. You provide general educational information only."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        add_log(f"Generated disease information for {disease_name}")

        return response.choices[0].message.content

    except Exception as e:
        add_log(f"Error while generating disease information: {str(e)}")
        return f"Error while generating disease information: {str(e)}"


# ============================================================
# STEP 12: Simple agentic AI function
# ============================================================
# This is the main agent.
# It reads the user query and decides which tool/function to call.

def healthcare_agent(user_query, patient_name):
    if user_query is None or user_query.strip() == "":
        return "Please enter your query."

    query = user_query.lower()

    add_log(f"User query received: {user_query}")

    # Tool 1: Retrieve history
    if "history" in query or "record" in query or "details" in query:
        return get_patient_history(patient_name)

    # Tool 2: Summarize patient history
    elif "summarize" in query or "summary" in query:
        return summarize_patient_history(patient_name)

    # Tool 3: Appointment booking guidance
    elif "appointment" in query or "book" in query or "doctor" in query:
        return "Please go to the 'Book Appointment' tab to book a doctor appointment."

    # Tool 4: Disease information
    elif "disease" in query or "symptom" in query or "treatment" in query or "medicine" in query:
        return disease_information(user_query)

    # General healthcare response
    else:
        if client is None:
            return key_missing_message()

        prompt = f"""
User asked: {user_query}

Respond as a helpful healthcare assistant.

Rules:
- Keep the answer simple.
- Do not give final medical diagnosis.
- Do not prescribe medicines.
- Suggest consulting a qualified doctor if needed.
"""

        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": "You are a safe healthcare assistant."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            add_log("Generated general healthcare response")

            return response.choices[0].message.content

        except Exception as e:
            add_log(f"Error while generating general healthcare response: {str(e)}")
            return f"Error while generating response: {str(e)}"


# ============================================================
# STEP 13: Show agent logs
# ============================================================

def show_logs():
    if len(agent_logs) == 0:
        return "No logs yet."

    return "\n".join(agent_logs)


# ============================================================
# STEP 14: Simple evaluation function
# ============================================================
# This is a basic evaluation screen.
# It is not a full QAEvalChain, but it gives a simple capstone-friendly
# evaluation view.

def show_evaluation_metrics():
    total_patients = len(patients_db)
    total_appointments = len(appointments_db)
    total_logs = len(agent_logs)

    return f"""
Simple Evaluation Metrics

Total Patient Records: {total_patients}
Total Appointments Booked: {total_appointments}
Total Agent Actions Logged: {total_logs}

Tool Success View:
- Patient record tool: Available
- Appointment booking tool: Available
- Patient history retrieval tool: Available
- AI summary tool: Available
- Disease information tool: Available
- Agent logs: Available

Note:
This is a beginner-level evaluation module.
In a real project, you can add accuracy score, response relevance score, booking success rate, and user feedback score.
"""


# ============================================================
# STEP 15: Build Gradio UI
# ============================================================

with gr.Blocks() as app:

    gr.Markdown("# Agentic Healthcare Assistant using Gradio")
    gr.Markdown("""
This is a simple Google Colab project for a healthcare assistant.

It can:
- Add patient records
- Retrieve patient history
- Summarize patient history using AI
- Book doctor appointments
- Provide disease information
- Show agent logs
- Show simple evaluation metrics
""")

    # --------------------------------------------------------
    # TAB 0: API Key
    # --------------------------------------------------------

    with gr.Tab("API Key"):

        gr.Markdown("### Enter your OpenAI API key (skip if OPENAI_API_KEY is already set)")

        api_key_input = gr.Textbox(
            label="OpenAI API Key",
            type="password",
            placeholder="Paste your OpenAI API key here"
        )

        api_key_status = gr.Textbox(
            label="API Key Status"
        )

        api_key_button = gr.Button("Save API Key")

        api_key_button.click(
            save_api_key,
            inputs=api_key_input,
            outputs=api_key_status
        )

    # --------------------------------------------------------
    # TAB 1: Chat with Healthcare Agent
    # --------------------------------------------------------

    with gr.Tab("Chat with Healthcare Agent"):

        user_query = gr.Textbox(
            label="Enter your healthcare query",
            placeholder="Example: Please summarize my father's kidney disease history",
            lines=3
        )

        chat_patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Example: Ramesh"
        )

        agent_output = gr.Textbox(
            label="Agent Response",
            lines=12
        )

        agent_button = gr.Button("Ask Agent")

        agent_button.click(
            healthcare_agent,
            inputs=[user_query, chat_patient_name],
            outputs=agent_output
        )

    # --------------------------------------------------------
    # TAB 2: Add Patient Record
    # --------------------------------------------------------

    with gr.Tab("Add Patient Record"):

        patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Example: Ramesh"
        )

        age = gr.Number(
            label="Age",
            value=70
        )

        condition = gr.Textbox(
            label="Medical Condition",
            placeholder="Example: Chronic Kidney Disease"
        )

        history = gr.Textbox(
            label="Medical History",
            lines=6,
            placeholder="Example: Patient has diabetes, high BP, and kidney issues for the last 5 years."
        )

        record_output = gr.Textbox(
            label="Status",
            lines=3
        )

        save_button = gr.Button("Save Patient Record")

        save_button.click(
            add_patient_record,
            inputs=[patient_name, age, condition, history],
            outputs=record_output
        )

    # --------------------------------------------------------
    # TAB 3: View Patient History
    # --------------------------------------------------------

    with gr.Tab("View Patient History"):

        history_patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Example: Ramesh"
        )

        history_output = gr.Textbox(
            label="Patient History",
            lines=10
        )

        history_button = gr.Button("Get Patient History")

        history_button.click(
            get_patient_history,
            inputs=history_patient_name,
            outputs=history_output
        )

    # --------------------------------------------------------
    # TAB 4: Summarize Patient History
    # --------------------------------------------------------

    with gr.Tab("Summarize Patient History"):

        summary_patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Example: Ramesh"
        )

        summary_output = gr.Textbox(
            label="Patient Summary",
            lines=12
        )

        summary_button = gr.Button("Summarize History")

        summary_button.click(
            summarize_patient_history,
            inputs=summary_patient_name,
            outputs=summary_output
        )

    # --------------------------------------------------------
    # TAB 5: Book Appointment
    # --------------------------------------------------------

    with gr.Tab("Book Appointment"):

        appointment_patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Example: Ramesh"
        )

        doctor_type = gr.Dropdown(
            choices=[
                "General Physician",
                "Cardiologist",
                "Nephrologist",
                "Neurologist",
                "Dermatologist",
                "Orthopedic Doctor",
                "Pediatrician",
                "ENT Specialist"
            ],
            label="Doctor Type",
            value="General Physician"
        )

        preferred_date = gr.Textbox(
            label="Preferred Date",
            placeholder="Example: 2026-06-10"
        )

        appointment_output = gr.Textbox(
            label="Appointment Status",
            lines=8
        )

        book_button = gr.Button("Book Appointment")

        book_button.click(
            book_appointment,
            inputs=[appointment_patient_name, doctor_type, preferred_date],
            outputs=appointment_output
        )

    # --------------------------------------------------------
    # TAB 6: View Appointments
    # --------------------------------------------------------

    with gr.Tab("View Appointments"):

        appointments_output = gr.Textbox(
            label="Appointments",
            lines=12
        )

        view_appointments_button = gr.Button("Show Appointments")

        view_appointments_button.click(
            show_appointments,
            inputs=[],
            outputs=appointments_output
        )

    # --------------------------------------------------------
    # TAB 7: Disease Information
    # --------------------------------------------------------

    with gr.Tab("Disease Information"):

        disease_name = gr.Textbox(
            label="Enter Disease Name or Medical Topic",
            placeholder="Example: Chronic Kidney Disease"
        )

        disease_output = gr.Textbox(
            label="Disease Information",
            lines=14
        )

        disease_button = gr.Button("Get Disease Information")

        disease_button.click(
            disease_information,
            inputs=disease_name,
            outputs=disease_output
        )

    # --------------------------------------------------------
    # TAB 8: Agent Logs
    # --------------------------------------------------------

    with gr.Tab("Agent Logs"):

        logs_output = gr.Textbox(
            label="Agent Logs",
            lines=16
        )

        logs_button = gr.Button("Show Logs")

        logs_button.click(
            show_logs,
            inputs=[],
            outputs=logs_output
        )

    # --------------------------------------------------------
    # TAB 9: Evaluation Metrics
    # --------------------------------------------------------

    with gr.Tab("Evaluation Metrics"):

        evaluation_output = gr.Textbox(
            label="Evaluation Metrics",
            lines=14
        )

        evaluation_button = gr.Button("Show Evaluation Metrics")

        evaluation_button.click(
            show_evaluation_metrics,
            inputs=[],
            outputs=evaluation_output
        )


# ============================================================
# STEP 16: Launch Gradio App
# ============================================================

app.launch(share=True)